# Credit Risk Modeling and Strategy Analysis

This notebook contains the complete workflow for the credit risk modeling project, including:

- Data loading and preprocessing
- Feature engineering
- One-hot encoding
- Feature selection using XGBoost
- XGBoost modeling and hyperparameter tuning
- SHAP explainability analysis
- Neural Network modeling
- Strategy evaluation and threshold analysis


In [ ]:
!pip install xgboost scikit-learn

import pandas as pd

# File paths
labels_path = "C:\\Users\\Desktop\\AML\\train_labels.csv"
data_path = "C:\\Users\\Desktop\\AML\\train_data.csv"

# Output paths
output_data_path = "C:\\Users\\Desktop\\AML\\dev_train_data.csv"
output_labels_path = "C:\\Users\\Desktop\\AML\\dev_train_labels.csv"

# Step 1: Load train_labels and sample 20%
train_labels = pd.read_csv(labels_path)
sampled_labels = train_labels.sample(frac=0.2, random_state=42)
sampled_ids = set(sampled_labels['customer_ID'])

# Step 2: Filter train_data.csv using chunking
chunks = []
chunk_size = 500_000  # Tune based on available memory

for chunk in pd.read_csv(data_path, chunksize=chunk_size):
    filtered_chunk = chunk[chunk['customer_ID'].isin(sampled_ids)]
    chunks.append(filtered_chunk)

# Step 3: Combine all filtered chunks
sampled_data = pd.concat(chunks, ignore_index=True)

# Step 4: Left join with sampled labels on customer_ID
merged_data = sampled_data.merge(sampled_labels, on='customer_ID', how='left')

# Step 5: Save results
merged_data.to_csv(output_data_path, index=False)
sampled_labels.to_csv(output_labels_path, index=False)

print("Sampled data created and saved successfully.")


In [ ]:
import pandas as pd

# Load only the first 5 rows
df_preview = pd.read_csv("C:\\Users\\Desktop\\AML\\dev_train_data.csv", nrows=5)

# Show structure
print("Shape:", df_preview.shape)
print("\nColumns:", df_preview.columns.tolist())
print("\nSample rows:")
print(df_preview.head())

# Check if 'target' is present
if 'target' in df_preview.columns:
    print("\n✅ 'target' column is present.")
else:
    print("\n❌ 'target' column is missing.")


In [ ]:
import pandas as pd

# Step 1: Read just the header to get number of columns
#df_columns = pd.read_csv("C:\\Users\\Desktop\\AML\\dev_train_data.csv", nrows=0)
num_columns = len(df_preview.columns)

# Step 2: Count rows using a generator
row_count = sum(1 for _ in open("C:\\Users\\Desktop\\AML\\dev_train_data.csv", 'r')) - 1  # subtract header

# Output
print(f"The dataset has {row_count:,} rows and {num_columns:,} columns.")


In [ ]:
import pandas as pd

# Step 1: Load merged dev data
df = pd.read_csv("C:\\Users\\Desktop\\AML\\dev_train_data.csv")

# Step 2: Convert date column to datetime format
df['S_2'] = pd.to_datetime(df['S_2'])

# Step 3: Categorical variables to be one-hot encoded
categorical_vars = ['B_30', 'B_38', 'D_114', 'D_116', 'D_117', 
                    'D_120', 'D_126', 'D_63', 'D_64', 'D_66', 'D_68']

# Step 4: Convert to string (in case some are numeric)
df[categorical_vars] = df[categorical_vars].astype(str)

# Step 5: One-hot encode in-place (adds new columns, drops originals)
df = pd.get_dummies(df, columns=categorical_vars, dummy_na=True)

# Now df includes one-hot encoded features and is ready for aggregation
print("One-hot encoding complete. Encoded shape:", df.shape)


In [ ]:
encoded_columns = [col for col in df.columns if any(var + "_" in col for var in categorical_vars)]

# Step 6: Extract and save to CSV
additional_columns = ['customer_ID', 'target']  # Replace these with actual column names you want to include

# Combine the encoded columns with the additional columns
columns_to_include = encoded_columns + additional_columns

df[columns_to_include].to_csv("C:\\Users\\Desktop\\AML\\one_hot_encoded_only.csv", index=False)

df.to_csv("C:\\Users\\Desktop\\AML\\final_data.csv", index=False)

print("One-hot encoded columns saved successfully.")


In [ ]:
import pandas as pd

# Load the dev train data
file_path = "C:\\Users\\Desktop\\AML\\final_data.csv"
df = pd.read_csv(file_path)

# Filter columns starting with 'B_38'
b30_cols = [col for col in df.columns if col.startswith("B_30")]

# Ensure target is included
cols_to_keep = ["customer_ID", "target"] + b30_cols

# Drop duplicate customers (assuming one row per customer is enough for this output)
unique_customers = df.drop_duplicates(subset="customer_ID")

# Sample 5 customer IDs
sample_ids = unique_customers["customer_ID"].sample(5, random_state=42)

# Filter data for those customer IDs
sample_data = unique_customers[unique_customers["customer_ID"].isin(sample_ids)][cols_to_keep]

# Display the result
print(" Sample of B_30 columns + target for 5 customers:")
print(sample_data)

# Optional: Save to CSV
# sample_data.to_csv("C:/Users/ganes/Desktop/amex_data/sample_B38_with_target.csv", index=False)



In [ ]:
import pandas as pd
import numpy as np

# -----------------------------------------
# Step 0: Ensure date is in datetime format
# -----------------------------------------
df = pd.read_csv("C:\\Users\\Desktop\\AML\\final_data.csv")
df['S_2'] = pd.to_datetime(df['S_2'])

print(df['S_2'].min(), df['S_2'].max())

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------------------
# Step 0: Ensure date is in datetime format
# -----------------------------------------
df = pd.read_csv("C:\\Users\\Desktop\\AML\\final_data.csv")
df['S_2'] = pd.to_datetime(df['S_2'])
df.sort_values(['customer_ID', 'S_2'], inplace=True)

# -----------------------------------------
# Step 1: Setup
# -----------------------------------------
id_col = 'customer_ID'
date_col = 'S_2'
target_col = 'target' if 'target' in df.columns else None

exclude_cols = [id_col, date_col]
if target_col:
    exclude_cols.append(target_col)

# -----------------------------------------
# Step 2: Identify feature types (Safe version)
# -----------------------------------------
numerical_features = []
categorical_features = []

for col in df.columns:
    if col in exclude_cols:
        continue
    if isinstance(col, str):
        col_series = df.loc[:, col]
        if isinstance(col_series, pd.Series):
            try:
                unique_vals = col_series.dropna().unique()
                if pd.api.types.is_numeric_dtype(col_series) and len(unique_vals) > 2:
                    numerical_features.append(col)
                elif set(unique_vals).issubset({0, 1}):
                    categorical_features.append(col)
            except Exception as e:
                print(f"⚠ Skipping column '{col}' due to error: {e}")
        else:
            print(f"⚠ Skipping column '{col}' — not a Series.")

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

# -----------------------------------------
# Step 3: Fast numerical aggregations
# -----------------------------------------
agg_funcs = ['min', 'max', 'median', 'std']
numerical_agg = df.groupby('customer_ID')[numerical_features].agg(agg_funcs)
numerical_agg.columns = [f"{col}_{func}" for col, func in numerical_agg.columns]
numerical_agg.reset_index(inplace=True)

# -----------------------------------------
# Step 4: Add rolling averages (3 and 6 months)
# -----------------------------------------
# Sort by date first to ensure rolling function works correctly
df.sort_values(['customer_ID', 'S_2'], inplace=True)

last3 = df[(df["S_2"] >= "2018-01-01") & (df["S_2"] <= "2018-04-01")]
last6 = df[(df["S_2"] >= "2017-10-01") & (df["S_2"] <= "2018-04-01")]

# Rolling 3-month average
rolling_3_mean = last3.groupby("customer_ID")[numerical_features].mean().reset_index()
rolling_3_mean.columns = ["customer_ID"] + [f"{col}_avg_3" for col in numerical_features]

# Rolling 6-month average
rolling_6_mean = last6.groupby("customer_ID")[numerical_features].mean().reset_index()
rolling_6_mean.columns = ["customer_ID"] + [f"{col}_avg_6" for col in numerical_features]

# -----------------------------------------
# Step 5: Add April 2018 values
# -----------------------------------------
apr_2018 = df[df['S_2'] == '2018-04-01'][['customer_ID'] + numerical_features].copy()
apr_2018 = apr_2018.rename(columns={col: f'{col}_apr_2018' for col in numerical_features})

merged = numerical_agg.merge(apr_2018, on='customer_ID', how='left')

# -----------------------------------------
# Step 6: Categorical aggregations (unchanged)
# -----------------------------------------
def aggregate_categoricals(group):
    features = {}
    for col in categorical_features:
        values = group[col]

        # Last 3 months % of True
        last3 = values.tail(3)
        if last3.count() > 0:
            features[f'{col}_pct_true_3'] = last3.sum() / last3.count()
        else:
            features[f'{col}_pct_true_3'] = None

        # Last 6 months % of True
        last6 = values.tail(6)
        if last6.count() > 0:
            features[f'{col}_pct_true_6'] = last6.sum() / last6.count()
        else:
            features[f'{col}_pct_true_6'] = None

    return pd.Series(features)

print("Aggregating categorical features...")
categorical_agg = df.groupby('customer_ID').apply(aggregate_categoricals).reset_index()

# -----------------------------------------
# Step 7: Merge and add target (unchanged)
# -----------------------------------------
final_df = merged.merge(categorical_agg, on='customer_ID', how='left')

if target_col:
    target_data = df[[id_col, target_col]].drop_duplicates()
    final_df = final_df.merge(target_data, on='customer_ID', how='left')

final_df1 = final_df.merge(rolling_3_mean, on='customer_ID', how='left')
final_df2 = final_df1.merge(rolling_6_mean, on='customer_ID', how='left')

# -----------------------------------------
# Step 8: Save to file (unchanged)
# -----------------------------------------
final_df2.to_csv("C:\\Users\\Desktop\\AML\\Step6.csv", index=False)
print("Step 6 complete — saved to 'Step6.csv'")


In [ ]:
import pandas as pd

# Load the final Step 6 data
df = pd.read_csv("C:\\Users\\Desktop\\AML\\Step6.csv")

# 1. Shape of the dataset
print(" Shape:")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("-" * 40)

# 2. Column data types
print(" Column Data Types:")
print(df.dtypes.value_counts())  # How many int, float, object, etc.
print("-" * 40)

# 3. Show a summary of column types
print(" Sample of data types:")
print(df.dtypes.head(10))  # See first 10 column types
print("-" * 40)

# 4. Preview the data
print(" Sample Data:")
display(df.head())  # Use print(df.head()) if not in Jupyter
print("target" in df.columns)

In [ ]:
import pandas as pd

# Load your development dataset (before aggregation)
df = pd.read_csv("C:\\Users\\Desktop\\AML\\dev_train_data.csv")
df['S_2'] = pd.to_datetime(df['S_2'])

# Count number of monthly records per customer
month_counts = df.groupby('customer_ID')['S_2'].nunique().reset_index()
month_counts.columns = ['customer_ID', 'num_months']

# Load target values (one row per customer)
targets = df[['customer_ID', 'target']].drop_duplicates()

# Merge to get num_months + target
merged = month_counts.merge(targets, on='customer_ID', how='left')

# Group by num_months to count observations and compute default rate
summary = merged.groupby('num_months').agg(
    Observations=('customer_ID', 'count'),
    Default_Rate=('target', 'mean')
).reset_index()

# Format default rate as percentage
summary['Default_Rate'] = (summary['Default_Rate'] * 100).round(2)

# Sort by number of months descending (optional)
summary = summary.sort_values('num_months', ascending=False)

# Show the table
print(summary.to_string(index=False))


In [ ]:
import pandas as pd

# Load the CSV file
file_path = r"C:\\Users\\Desktop\\AML\\Step6.csv"  # Update this if needed
df = pd.read_csv(file_path)

# Print all column names
print("Total columns:", len(df.columns))
for i, col in enumerate(df.columns):
    print(f"{i+1:>3}: {col}")


In [ ]:
import pandas as pd
from collections import defaultdict

# Load the file
file_path = r"C:\\Users\\Desktop\\AML\\Step6.csv"
df = pd.read_csv(file_path)
all_cols = df.columns.tolist()

# Define your aggregation keywords
keywords = {
    '_min': 'Minimum',
    '_max': 'Maximum',
    '_median': 'Median',
    '_std': 'Standard Deviation',
    '_avg_3': 'Average (last 3 months)',
    '_avg_6': 'Average (last 6 months)',
    '_apr_2018': 'April 2018'
}

# Count occurrences
feature_type_counts = defaultdict(int)
for col in all_cols:
    col_lower = col.lower()
    for key in keywords:
        if key in col_lower:
            feature_type_counts[keywords[key]] += 1

# Convert to DataFrame and print
feature_counts_df = pd.DataFrame(list(feature_type_counts.items()), columns=["Feature Type", "Count"])
print(feature_counts_df)



In [ ]:
import pandas as pd

# Load the Step6 file
df = pd.read_csv(r"C:\\Users\\Desktop\\AML\\Step6.csv")

# Filter columns starting with 'P_2'
p2_columns = [col for col in df.columns if col.startswith('P_2')]

# Get data types of those columns
p2_dtypes = df[p2_columns].dtypes

# Display nicely
print(f"Found {len(p2_dtypes)} columns starting with 'P_2':\n")
for col, dtype in p2_dtypes.items():
    print(f"{col:30} → {dtype}")


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Step 1: Load your final engineered dataset
df = pd.read_csv("C:\\Users\\Desktop\\AML\\Step6.csv")

# Step 2: Split 70% Train, 30% Temp (for Test1 and Test2)
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['target'])

# Step 3: Split 30% Temp into 15% Test1 and 15% Test2 (i.e., 50/50 of remaining)
test1_df, test2_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['target'])

# Step 4: Save (optional)
train_df.to_csv("C:\\Users\\Desktop\\AML\\train_set.csv", index=False)
test1_df.to_csv("C:\\Users\\Desktop\\AML\\test1_set.csv", index=False)
test2_df.to_csv("C:\\Users\\Desktop\\AML\\test2_set.csv", index=False)

# Step 5: Check sizes
print(f"Train size: {len(train_df)}")
print(f"Test1 size: {len(test1_df)}")
print(f"Test2 size: {len(test2_df)}")


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier

# Load your train set
df = pd.read_csv("C:\\Users\\Desktop\\AML\\train_set.csv")

# Clean up infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop unnecessary columns
X = df.drop(columns=['customer_ID', 'target'])
y = df['target']

# ----------------------
# Model 1: Default XGBoost
# ----------------------
model1 = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model1.fit(X, y)

importance1 = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model1.feature_importances_
})
importance1.to_csv("C:\\Users\\Desktop\\AML\\xgb_feature_importance_default.csv", index=False)

# ----------------------
# Model 2: Custom XGBoost
# ----------------------
model2 = XGBClassifier(
    n_estimators=300,
    learning_rate=0.5,
    max_depth=4,
    subsample=0.5,
    colsample_bytree=0.5,
    scale_pos_weight=5,
    use_label_encoder=False,
    eval_metric='logloss'
)
model2.fit(X, y)

importance2 = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model2.feature_importances_
})
importance2.to_csv("C:\\Users\\Desktop\\AML\\xgb_feature_importance_custom.csv", index=False)

# ----------------------
# Select features with >0.5% importance in either model
# ----------------------
selected_features = set(
    importance1[importance1['Importance'] > 0.005]['Feature']
).union(
    importance2[importance2['Importance'] > 0.005]['Feature']
)

# Save selected features
pd.DataFrame({'Selected_Features': list(selected_features)}).to_csv(
    "C:\\Users\\Desktop\\AML\\selected_features.csv", index=False)

# Filtered train data for next step
X_selected = X[list(selected_features)]
X_selected.to_csv("C:\\Users\\Desktop\\AML\\train_selected_features.csv", index=False)
y.to_csv("C:\\Users\\Desktop\\AML\\train_target.csv", index=False)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load CSVs
default_imp = pd.read_csv("C:\\Users\\Desktop\\AML\\xgb_feature_importance_default.csv")
custom_imp = pd.read_csv("C:\\Users\\Desktop\\AML\\xgb_feature_importance_custom.csv")

# Merge and compute max importance
merged = pd.merge(default_imp, custom_imp, on='Feature', how='outer', suffixes=('_default', '_custom')).fillna(0)
merged['Importance'] = merged[['Importance_default', 'Importance_custom']].max(axis=1)

# Filter features with importance > 0.005
important_features = merged[merged['Importance'] > 0.005].copy()
important_features = important_features.sort_values(by='Importance', ascending=True)  # for horizontal plot

# Save to CSV
important_features[['Feature', 'Importance']].to_csv("C:\\Users\\Desktop\\AML\\importance_for_graph.csv", index=False)

# Plot: horizontal bar chart
plt.figure(figsize=(10, 8))
bars = plt.barh(important_features['Feature'], important_features['Importance'], color='skyblue')
plt.xlabel("Importance")
plt.title("XGBoost Selected Features (Importance > 0.5%)")

# Add importance labels
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.001, bar.get_y() + bar.get_height()/2, f'{width:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import itertools
import os
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# ----------------------------------
# Load Training and Target Data
# ----------------------------------
X = pd.read_csv("C:\\Users\\Desktop\\AML\\train_selected_features.csv")
y = pd.read_csv("C:\\Users\\Desktop\\AML\\train_target.csv").squeeze()

# ----------------------------------
# Load Test1 and Test2
# ----------------------------------
test1 = pd.read_csv("C:\\Users\\Desktop\\AML\\test1_set.csv")
test2 = pd.read_csv("C:\\Users\\Desktop\\AML\\test2_set.csv")

# Filter to use only selected features
X_test1 = test1[X.columns]
y_test1 = test1['target']
X_test2 = test2[X.columns]
y_test2 = test2['target']

# ----------------------------------
# Grid Parameters
# ----------------------------------
n_trees = [50, 100, 300]
learning_rates = [0.01, 0.1]
subsamples = [0.5, 0.8]
colsamples = [0.5, 1.0]
weights = [1, 5, 10]

# Results setup
results_path = "C:\\Users\\Desktop\\AML\\xgb_grid_results.csv"

if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
else:
    results_df = pd.DataFrame(columns=[
        "Trees", "LR", "Subsample", "% Features", "Weight",
        "AUC_Train", "AUC_Test1", "AUC_Test2"
    ])

# ----------------------------------
# Grid Search Loop
# ----------------------------------
for n, lr, sub, col, w in itertools.product(n_trees, learning_rates, subsamples, colsamples, weights):
    
    already_done = (
        (results_df["Trees"] == n) &
        (results_df["LR"] == lr) &
        (results_df["Subsample"] == sub) &
        (results_df["% Features"] == col) &
        (results_df["Weight"] == w)
    )

    if already_done.any():
        continue  # Skip already completed combinations

    print(f" Training: Trees={n}, LR={lr}, Subsample={sub}, Colsample={col}, Weight={w}")

    # Train model
    model = XGBClassifier(
        n_estimators=n,
        learning_rate=lr,
        subsample=sub,
        colsample_bytree=col,
        scale_pos_weight=w,
        max_depth=4,
        eval_metric='logloss',
        verbosity=0
    )
    model.fit(X, y)

    # Evaluate
    auc_train = roc_auc_score(y, model.predict_proba(X)[:, 1])
    auc_test1 = roc_auc_score(y_test1, model.predict_proba(X_test1)[:, 1])
    auc_test2 = roc_auc_score(y_test2, model.predict_proba(X_test2)[:, 1])

    # Save result
    new_row = {
        "Trees": n,
        "LR": lr,
        "Subsample": sub,
        "% Features": col,
        "Weight": w,
        "AUC_Train": round(auc_train, 4),
        "AUC_Test1": round(auc_test1, 4),
        "AUC_Test2": round(auc_test2, 4),
    }

    # ✅ Compatible with pandas ≥ 2.0
    results_df.loc[len(results_df)] = new_row

    # Save progress
    results_df.to_csv(results_path, index=False)
    print("✅ Saved to file.")


In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import joblib

# --------------------------------------
# Load Best Model from Grid Results (Bias-Variance Based)
# --------------------------------------
results_path = "C:\\Users\\Desktop\\AML\\xgb_grid_results.csv"
results = pd.read_csv(results_path)

# Compute bias and variance
results['bias'] = 1 - results['AUC_Test1']
results['variance'] = abs(results['AUC_Train'] - results['AUC_Test1'])

# Select best model based on lowest bias and variance
best_row = results.sort_values(by=['bias', 'variance'], ascending=True).iloc[0]

print(" Best Bias-Variance Balanced Model:\n", best_row)

# --------------------------------------
# Load Data
# --------------------------------------
X_train = pd.read_csv("C:\\Users\\Desktop\\AML\\train_selected_features.csv")
y_train = pd.read_csv("C:\\Users\\Desktop\\AML\\train_target.csv").squeeze()

test1 = pd.read_csv("C:\\Users\\Desktop\\AML\\test1_set.csv")
test2 = pd.read_csv("C:\\Users\\Desktop\\AML\\test2_set.csv")

# Use same feature columns as training
X_test1 = test1[X_train.columns]
y_test1 = test1['target']
X_test2 = test2[X_train.columns]
y_test2 = test2['target']

# --------------------------------------
# Train Final Model on Train Only
# --------------------------------------
final_model = XGBClassifier(
    n_estimators=int(best_row['Trees']),
    learning_rate=float(best_row['LR']),
    subsample=float(best_row['Subsample']),
    colsample_bytree=float(best_row['% Features']),
    scale_pos_weight=float(best_row['Weight']),
    max_depth=4,
    eval_metric='logloss',
    verbosity=0
)

print(" Training final model on Train...")
final_model.fit(X_train, y_train)

# --------------------------------------
# Save the Final Model
# --------------------------------------
model_path = "C:\\Users\\Desktop\\AML\\final_model.xgb"
joblib.dump(final_model, model_path)
print(f"✅ Final model saved to: {model_path}")

# --------------------------------------
# Evaluate on Test1 and Test2
# --------------------------------------
auc_test1 = roc_auc_score(y_test1, final_model.predict_proba(X_test1)[:, 1])
auc_test2 = roc_auc_score(y_test2, final_model.predict_proba(X_test2)[:, 1])

print(" Final AUC on Test1:", round(auc_test1, 4))
print(" Final AUC on Test2:", round(auc_test2, 4))


In [ ]:
import shap
import pandas as pd
import joblib
import matplotlib.pyplot as plt

# Load model
model = joblib.load("C:\\Users\\Desktop\\AML\\final_model.xgb")

# Load Test2 set
test2 = pd.read_csv("C:\\Users\\Desktop\\AML\\test2_set.csv")
y_test2 = test2["target"]
X_train = pd.read_csv("C:\\Users\\Desktop\\AML\\train_selected_features.csv")
X_test2 = test2[X_train.columns]

# Pick one customer (e.g., index 5)
index = 5
sample = X_test2.iloc[[index]]  # Double brackets to keep it a DataFrame

# Create explainer and SHAP values
explainer = shap.Explainer(model)
shap_values = explainer(X_test2)

# Plot waterfall for that customer
plt.figure()
shap.plots.waterfall(shap_values[index], max_display=15, show=False)
plt.title(f"SHAP Waterfall Plot – Test2 Sample #{index}")
plt.tight_layout()
plt.savefig("C:\\Users\\Desktop\\AML\\shap_waterfall_test2_obs.png")
plt.show()


In [ ]:
#NN
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

#  Paths
base_path = "C:\\Users\\Desktop\\AML\\"
selected_feat_path = base_path + "selected_features.csv"

#  Load selected features
selected = pd.read_csv(selected_feat_path)["Selected_Features"].tolist()

#  Load datasets
X_train = pd.read_csv(base_path + "train_selected_features.csv")[selected]
y_train = pd.read_csv(base_path + "train_target.csv").squeeze()
test1 = pd.read_csv(base_path + "test1_set.csv")
test2 = pd.read_csv(base_path + "test2_set.csv")

X_test1 = test1[selected]
y_test1 = test1["target"]
X_test2 = test2[selected]
y_test2 = test2["target"]

# ✅ Step 1: Impute missing values with 0
X_train = X_train.fillna(0)
X_test1 = X_test1.fillna(0)
X_test2 = X_test2.fillna(0)

# ✅ Step 2: Outlier Treatment (cap at 1st and 99th percentiles using train stats)
lower = X_train.quantile(0.01)
upper = X_train.quantile(0.99)

X_train = X_train.clip(lower, upper, axis=1)
X_test1 = X_test1.clip(lower, upper, axis=1)
X_test2 = X_test2.clip(lower, upper, axis=1)

# ✅ Step 3: Standardization (Z-score using StandardScaler on train only)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=selected)
X_test1_scaled = pd.DataFrame(scaler.transform(X_test1), columns=selected)
X_test2_scaled = pd.DataFrame(scaler.transform(X_test2), columns=selected)

# ✅ Step 4: Save preprocessed data
X_train_scaled.to_csv(base_path + "nn_train_X.csv", index=False)
y_train.to_csv(base_path + "nn_train_y.csv", index=False)
X_test1_scaled.to_csv(base_path + "nn_test1_X.csv", index=False)
y_test1.to_csv(base_path + "nn_test1_y.csv", index=False)
X_test2_scaled.to_csv(base_path + "nn_test2_X.csv", index=False)
y_test2.to_csv(base_path + "nn_test2_y.csv", index=False)

print("✅ Data preprocessing for Neural Network is complete.")


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.metrics import roc_auc_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
import itertools
import os

# === Load Data ===
base_path = "C:\\Users\\Desktop\\AML\\"

X_train = pd.read_csv(base_path + "nn_train_X.csv")
y_train = pd.read_csv(base_path + "nn_train_y.csv").squeeze()
X_test1 = pd.read_csv(base_path + "nn_test1_X.csv")
y_test1 = pd.read_csv(base_path + "nn_test1_y.csv").squeeze()
X_test2 = pd.read_csv(base_path + "nn_test2_X.csv")
y_test2 = pd.read_csv(base_path + "nn_test2_y.csv").squeeze()

# === Grid Parameters ===
hidden_layers = [2, 4]
nodes = [4, 6]
activations = ['relu', 'tanh']
dropouts = [0.5, 0.0]
batch_sizes = [100, 10000]

# Results CSV
results_path = base_path + "nn_grid_results.csv"
columns = ['# HL', '# Node', 'Activation', 'Dropout', 'Batch Size', 'AUC Train', 'AUC Test1', 'AUC Test2']

# Load previous results or create empty
if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
else:
    results_df = pd.DataFrame(columns=columns)

# === Model Builder ===
def build_model(input_dim, hl, nodes, activation, dropout):
    model = Sequential()
    model.add(Dense(nodes, activation=activation, input_dim=input_dim))
    if dropout > 0:
        model.add(Dropout(dropout))
    for _ in range(hl - 1):
        model.add(Dense(nodes, activation=activation))
        if dropout > 0:
            model.add(Dropout(dropout))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=Adam(), loss='binary_crossentropy')
    return model

# === Grid Search ===
for hl, n, act, do, bs in itertools.product(hidden_layers, nodes, activations, dropouts, batch_sizes):
    row_match = (
        (results_df['# HL'] == hl) &
        (results_df['# Node'] == n) &
        (results_df['Activation'] == act) &
        (results_df['Dropout'] == f"{int(do*100)}%") &
        (results_df['Batch Size'] == bs)
    )
    if row_match.any():
        continue  # Skip already done

    print(f" Training HL={hl}, Nodes={n}, Act={act}, Dropout={do}, Batch={bs}")
    model = build_model(X_train.shape[1], hl, n, act, do)
    model.fit(X_train, y_train, epochs=20, batch_size=bs, verbose=0)

    auc_train = roc_auc_score(y_train, model.predict(X_train).ravel())
    auc_test1 = roc_auc_score(y_test1, model.predict(X_test1).ravel())
    auc_test2 = roc_auc_score(y_test2, model.predict(X_test2).ravel())

    result = {
        '# HL': hl,
        '# Node': n,
        'Activation': act,
        'Dropout': f"{int(do*100)}%",
        'Batch Size': bs,
        'AUC Train': round(auc_train, 4),
        'AUC Test1': round(auc_test1, 4),
        'AUC Test2': round(auc_test2, 4),
    }

    results_df = pd.concat([results_df, pd.DataFrame([result])], ignore_index=True)
    results_df.to_csv(results_path, index=False)
    print(f"✅ Saved: AUC Test2 = {auc_test2:.4f}")


In [ ]:
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import roc_auc_score

# Step 1: Load the processed training and test sets
X_train = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_train_X.csv")
y_train = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_train_y.csv").squeeze()
X_test1 = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_test1_X.csv")
y_test1 = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_test1_y.csv").squeeze()
X_test2 = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_test2_X.csv")
y_test2 = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_test2_y.csv").squeeze()

# Step 2: Define best model architecture
def build_final_nn(input_dim):
    model = Sequential()
    model.add(Dense(6, activation='tanh', input_dim=input_dim))  # First hidden layer
    model.add(Dense(6, activation='tanh'))                        # Second hidden layer
    model.add(Dense(6, activation='tanh'))                        # Third hidden layer
    model.add(Dense(6, activation='tanh'))                        # Fourth hidden layer
    model.add(Dense(1, activation='sigmoid'))                     # Output layer
    model.compile(optimizer=Adam(), loss='binary_crossentropy')
    return model

# Step 3: Build and train the model
final_model = build_final_nn(X_train.shape[1])
final_model.fit(X_train, y_train, epochs=20, batch_size=100, verbose=1)

# Step 4: Save model
final_model.save("C:\\Users\\Desktop\\AML\\final_nn_model.h5")
print("✅ Final neural network model saved to final_nn_model.h5")

# Step 5: Evaluate model
train_preds = final_model.predict(X_train)
test1_preds = final_model.predict(X_test1)
test2_preds = final_model.predict(X_test2)

print("\n Final AUC Scores:")
print(f"AUC Train : {roc_auc_score(y_train, train_preds):.4f}")
print(f"AUC Test1 : {roc_auc_score(y_test1, test1_preds):.4f}")
print(f"AUC Test2 : {roc_auc_score(y_test2, test2_preds):.4f}")


In [ ]:
import pandas as pd
import joblib
import numpy as np
from tensorflow.keras.models import load_model
from sklearn.metrics import roc_auc_score

# ---------------------------
# Step 1: Load data
# ---------------------------
# For XGBoost → use selected features
X_xgb = pd.read_csv("C:\\Users\\Desktop\\AML\\train_selected_features.csv")
y_train = pd.read_csv("C:\\Users\\Desktop\\AML\\train_target.csv").squeeze()
test1 = pd.read_csv("C:\\Users\\Desktop\\AML\\test1_set.csv")
test2 = pd.read_csv("C:\\Users\\Desktop\\AML\\test2_set.csv")

X_test1_xgb = test1[X_xgb.columns]
y_test1 = test1["target"]
X_test2_xgb = test2[X_xgb.columns]
y_test2 = test2["target"]

# For Neural Network → use normalized NN input
X_nn = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_train_X.csv")
y_nn = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_train_y.csv").squeeze()
X_test1_nn = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_test1_X.csv")
y_test1_nn = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_test1_y.csv").squeeze()
X_test2_nn = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_test2_X.csv")
y_test2_nn = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_test2_y.csv").squeeze()

# ---------------------------
# Step 2: Load models
# ---------------------------
xgb_model = joblib.load("C:\\Users\\Desktop\\AML\\final_model.xgb")
nn_model = load_model("C:\\Users\\Desktop\\AML\\final_nn_model.h5")

# ---------------------------
# Step 3: Predictions (ravel + safe)
# ---------------------------
xgb_preds_train = xgb_model.predict_proba(X_xgb)[:, 1]
xgb_preds_test1 = xgb_model.predict_proba(X_test1_xgb)[:, 1]
xgb_preds_test2 = xgb_model.predict_proba(X_test2_xgb)[:, 1]

nn_preds_train = nn_model.predict(X_nn).ravel()
nn_preds_test1 = nn_model.predict(X_test1_nn).ravel()
nn_preds_test2 = nn_model.predict(X_test2_nn).ravel()

# ---------------------------
# Step 4: AUC Calculation
# ---------------------------
print("\n Final AUC Comparison:")

print("\n XGBoost:")
xgb_auc_train = roc_auc_score(y_train, xgb_preds_train)
xgb_auc_test1 = roc_auc_score(y_test1, xgb_preds_test1)
xgb_auc_test2 = roc_auc_score(y_test2, xgb_preds_test2)
print(f"AUC Train : {xgb_auc_train:.4f}")
print(f"AUC Test1 : {xgb_auc_test1:.4f}")
print(f"AUC Test2 : {xgb_auc_test2:.4f}")

print("\n Neural Network:")
nn_auc_train = roc_auc_score(y_nn, nn_preds_train)
nn_auc_test1 = roc_auc_score(y_test1_nn, nn_preds_test1)
nn_auc_test2 = roc_auc_score(y_test2_nn, nn_preds_test2)
print(f"AUC Train : {nn_auc_train:.4f}")
print(f"AUC Test1 : {nn_auc_test1:.4f}")
print(f"AUC Test2 : {nn_auc_test2:.4f}")

# ---------------------------
# Step 5: Winner
# ---------------------------
print("\n Model Selection Based on Test2 AUC:")
if nn_auc_test2 > xgb_auc_test2:
    print("✅ Neural Network performs better on unseen data (Test2).")
elif xgb_auc_test2 > nn_auc_test2:
    print("✅ XGBoost performs better on unseen data (Test2).")
else:
    print("⚖️ Both models perform equally on Test2.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the updated grid search file
df = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_grid_results_with_avg_std.csv")

# === Find best model for Plot 1: high AUC_Avg and low AUC_Std ===
best_plot1 = df.sort_values(by=["AUC_Avg", "AUC_Std"], ascending=[False, True]).iloc[0]

# === Find best model for Plot 2: high AUC Test2, not overfitting ===
df["Gap"] = df["AUC Train"] - df["AUC Test2"]
best_plot2 = df.loc[df["AUC Test2"] == df["AUC Test2"].max()].sort_values("Gap").iloc[0]

# === Plot 1: AUC Avg vs AUC Std Dev ===
plt.figure(figsize=(8, 6))
plt.scatter(df["AUC_Avg"], df["AUC_Std"], alpha=0.6, label="All Models")
plt.scatter(best_plot1["AUC_Avg"], best_plot1["AUC_Std"], color='red', label="Best Model", s=100)
plt.title("Scatter Plot: AUC Average vs Std Deviation")
plt.xlabel("Average AUC")
plt.ylabel("AUC Std Deviation")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("C:\\Users\\Desktop\\AML\\nn_plot_auc_avg_vs_std_labeled.png")
plt.show()

# === Plot 2: AUC Train vs AUC Test2 ===
plt.figure(figsize=(8, 6))
plt.scatter(df["AUC Train"], df["AUC Test2"], alpha=0.6, label="All Models")
plt.scatter(best_plot2["AUC Train"], best_plot2["AUC Test2"], color='red', label="Best Model", s=100)
plt.title("Scatter Plot: AUC Train vs AUC Test2")
plt.xlabel("AUC Train")
plt.ylabel("AUC Test2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("C:\\Users\\Desktop\\AML\\nn_plot_train_vs_test2_labeled.png")
plt.show()

# === Output Best Model Parameters ===
print("\n Best Model from Plot 1 (High AUC Avg, Low Std Dev):")
print(best_plot1[["# HL", "# Node", "Activation", "Dropout", "Batch Size", "AUC_Avg", "AUC_Std"]])

print("\n Best Model from Plot 2 (Best Generalization):")
print(best_plot2[["# HL", "# Node", "Activation", "Dropout", "Batch Size", "AUC Train", "AUC Test2", "Gap"]])


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from sklearn.metrics import roc_auc_score

# === Step 1: Load final trained Neural Network model ===
nn_model = load_model("C:\\Users\\Desktop\\AML\\final_nn_model.h5")

# === Step 2: Load processed datasets ===
X_train = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_train_X.csv")
y_train = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_train_y.csv").squeeze()
X_test1 = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_test1_X.csv")
y_test1 = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_test1_y.csv").squeeze()
X_test2 = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_test2_X.csv")
y_test2 = pd.read_csv("C:\\Users\\Desktop\\AML\\nn_test2_y.csv").squeeze()

# === Step 3: Predict default scores ===
train_scores = nn_model.predict(X_train).ravel()
test1_scores = nn_model.predict(X_test1).ravel()
test2_scores = nn_model.predict(X_test2).ravel()

# === Step 4: Calculate AUCs ===
auc_train = roc_auc_score(y_train, train_scores)
auc_test1 = roc_auc_score(y_test1, test1_scores)
auc_test2 = roc_auc_score(y_test2, test2_scores)

print(" AUC Scores for Final Neural Network Model")
print(f"AUC - Train : {auc_train:.4f}")
print(f"AUC - Test1 : {auc_test1:.4f}")
print(f"AUC - Test2 : {auc_test2:.4f}")

# === Step 5: Define score bins based on Train ===
bins = np.percentile(train_scores, q=np.linspace(0, 100, 11))  # 10 bins
labels = [f"{i+1}" for i in range(10)]

def rank_order(scores, target, bins, name):
    df = pd.DataFrame({"score": scores, "target": target})
    df["bin"] = pd.cut(df["score"], bins=bins, labels=labels, include_lowest=True)
    grouped = df.groupby("bin")["target"].mean().reset_index()
    grouped.columns = ["Score Bin", f"Default Rate - {name}"]
    return grouped

# === Step 6: Get rank order dataframes ===
rank_train = rank_order(train_scores, y_train, bins, "Train")
rank_test1 = rank_order(test1_scores, y_test1, bins, "Test1")
rank_test2 = rank_order(test2_scores, y_test2, bins, "Test2")

# === Step 7: Merge all into one DataFrame ===
rank_df = rank_train.merge(rank_test1, on="Score Bin").merge(rank_test2, on="Score Bin")
rank_df.to_csv("C:\\Users\\Desktop\\AML\\nn_rank_order_default_rates.csv", index=False)

# === Step 8: Plot Rank Ordering Chart ===
rank_df.set_index("Score Bin").astype(float).plot(kind="bar", figsize=(10, 6))
plt.title("Neural Network Rank Ordering: Default Rate by Score Bin")
plt.xlabel("Score Bin (1 = Lowest Risk, 10 = Highest Risk)")
plt.ylabel("Default Rate")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("C:\\Users\\Desktop\\AML\\nn_rank_order_plot.png")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import joblib

# === Step 1: Load your merged data ===
df = pd.read_csv("C:/Users/ganes/Desktop/AML/final_data.csv")
df["S_2"] = pd.to_datetime(df["S_2"])

# === Step 2: Filter for last 6 months (Nov 2017 to Apr 2018) ===
last6 = df[(df["S_2"] >= "2017-10-01") & (df["S_2"] <= "2018-04-01")]

# === Step 3: Calculate S_Ave and B_Ave ===
avg_df = last6.groupby("customer_ID")[["S_25", "B_1"]].mean().reset_index()
avg_df.columns = ["customer_ID", "S_Ave", "B_Ave"]

# === Step 4: Load your Step6 data to get target ===
step6 = pd.read_csv("C:/Users/ganes/Desktop/AML/Step6.csv")
targets = step6[["customer_ID", "target"]]

# === Step 5: Load selected features and model ===
selected_features = pd.read_csv("C:/Users/ganes/Desktop/AML/train_selected_features.csv", nrows=1).columns.tolist()
xgb_model = joblib.load("C:/Users/ganes/Desktop/AML/final_model.xgb")

X = step6[selected_features]
customer_ids = step6["customer_ID"]
y = step6["target"]
preds = xgb_model.predict_proba(X)[:, 1]

# === Step 6: Create predictions DataFrame ===
pred_df = pd.DataFrame({
    "customer_ID": customer_ids,
    "predicted_PD": preds
})

# === Step 7: Merge predictions + target + average spend/balance ===
final = pred_df.merge(targets, on="customer_ID").merge(avg_df, on="customer_ID")

# === Step 8: Save the clean strategy input file ===
final[["target", "predicted_PD", "B_Ave", "S_Ave"]].to_csv(
    "C:/Users/ganes/Desktop/AML/strategy_input_data-Full(Step6).csv", index=False
)

print("✅ Saved strategy_input_data.csv with 4 required columns.")


In [ ]:
#Step 15
import pandas as pd
import numpy as np

# === Step 1: Load strategy input data ===
data = pd.read_csv("C:/Users/ganes/Desktop/AML/strategy_input_data-Full(Step6).csv")

# === Step 2: Define the strategy function ===
def calculate_default_and_revenue(df, target_col, pd_col, balance_col, spend_col, threshold):
    accepted = df[df[pd_col] < threshold]
    if accepted.empty:
        return 0.0, 0.0

    default_rate = accepted[target_col].mean()
    non_defaulters = accepted[accepted[target_col] == 0]

    monthly_revenue = non_defaulters[balance_col] * 0.02 + non_defaulters[spend_col] * 0.001
    annual_revenue = monthly_revenue.sum() * 12

    return round(default_rate, 4), round(annual_revenue, 2)

# === Step 3: Loop through thresholds ===
thresholds = np.arange(0.01, 0.151, 0.01)
results = []

for t in thresholds:
    d_rate, revenue = calculate_default_and_revenue(
        data,
        target_col="target",
        pd_col="predicted_PD",
        balance_col="B_Ave",
        spend_col="S_Ave",
        threshold=t
    )
    results.append({
        "Threshold": round(t, 2),
        "Default Rate": d_rate,
        "Expected Revenue": revenue
    })

# === Step 4: Create results DataFrame ===
strategy_df = pd.DataFrame(results)

# Identify conservative and aggressive strategies (<= 10% default)
conservative = strategy_df[strategy_df["Default Rate"] <= 0.10].iloc[0]
aggressive = strategy_df[strategy_df["Default Rate"] <= 0.10].iloc[-1]

# Save results
strategy_df.to_csv("C:/Users/ganes/Desktop/AML/strategy_threshold_results-Ste6(1-15).csv", index=False)

# Print results
print("✅ Strategy analysis complete.")
print("\n Conservative Strategy:\n", conservative)
print("\n Aggressive Strategy:\n", aggressive)


In [ ]:
import pandas as pd
import numpy as np

# === Step 1: Load input data (train sample only) ===
# This file must include: 'target', 'predicted_PD', 'B_Ave', 'S_Ave'
data = pd.read_csv("C:/Users/ganes/Desktop/AML/strategy_input_data-Full(Step6).csv")

# === Step 2: Define evaluation function ===
def evaluate_strategy(data, target_col, pd_col, balance_col, spend_col, threshold):
    """
    Returns default rate and expected annual revenue for a given threshold.
    """
    accepted = data[data[pd_col] < threshold]
    
    if accepted.empty:
        return 0.0, 0.0

    default_rate = accepted[target_col].mean()

    # Revenue only from non-defaulters
    non_defaults = accepted[accepted[target_col] == 0]
    monthly_revenue = non_defaults[balance_col] * 0.02 + non_defaults[spend_col] * 0.001
    annual_revenue = monthly_revenue.sum() * 12

    return round(default_rate, 4), round(annual_revenue, 2)

# === Step 3: Loop through thresholds and collect results ===
thresholds = np.arange(0.1, 1.01, 0.1)
results = []

for t in thresholds:
    d_rate, revenue = evaluate_strategy(
        data=data,
        target_col="target",
        pd_col="predicted_PD",
        balance_col="B_Ave",
        spend_col="S_Ave",
        threshold=t
    )
    results.append({
        "Threshold": round(t, 2),
        "Default Rate": d_rate,
        "Expected Revenue": revenue
    })

# === Step 4: Convert to DataFrame and filter valid strategies ===
results_df = pd.DataFrame(results)
valid = results_df[results_df["Default Rate"] <= 0.10]

# Pick strategies
conservative = valid.iloc[0] if not valid.empty else None
aggressive = valid.iloc[-1] if not valid.empty else None

# === Step 5: Save results ===
results_df.to_csv("C:/Users/ganes/Desktop/AML/strategy_threshold_results(Step6 (1-100)).csv", index=False)

# === Step 6: Print summary ===
print("\n Strategy Threshold Evaluation")
print(results_df)

print("\n✅ Conservative Strategy (First with default ≤ 10%):")
print(conservative)

print("\n✅ Aggressive Strategy (Last with default ≤ 10%):")
print(aggressive)


In [ ]:
import pandas as pd
import numpy as np
import joblib

# === Step 1: Load your merged data ===
df = pd.read_csv("C:/Users/ganes/Desktop/AML/final_data.csv")
df["S_2"] = pd.to_datetime(df["S_2"])

# === Step 2: Filter for last 6 months (Nov 2017 to Apr 2018) ===
last6 = df[(df["S_2"] >= "2017-10-01") & (df["S_2"] <= "2018-04-01")]

# === Step 3: Calculate S_Ave and B_Ave ===
avg_df = last6.groupby("customer_ID")[["S_25", "B_1"]].mean().reset_index()
avg_df.columns = ["customer_ID", "S_Ave", "B_Ave"]

# === Step 4: Load your Step6 data to get target ===
step6 = pd.read_csv("C:/Users/ganes/Desktop/AML/train_set.csv")
targets = step6[["customer_ID", "target"]]

# === Step 5: Load selected features and model ===
selected_features = pd.read_csv("C:/Users/ganes/Desktop/AML/train_selected_features.csv", nrows=1).columns.tolist()
xgb_model = joblib.load("C:/Users/ganes/Desktop/AML/final_model.xgb")

X = step6[selected_features]
customer_ids = step6["customer_ID"]
y = step6["target"]
preds = xgb_model.predict_proba(X)[:, 1]

# === Step 6: Create predictions DataFrame ===
pred_df = pd.DataFrame({
    "customer_ID": customer_ids,
    "predicted_PD": preds
})

# === Step 7: Merge predictions + target + average spend/balance ===
final = pred_df.merge(targets, on="customer_ID").merge(avg_df, on="customer_ID")

# === Step 8: Save the clean strategy input file ===
final[["target", "predicted_PD", "B_Ave", "S_Ave"]].to_csv(
    "C:/Users/ganes/Desktop/AML/strategy_input_data-Train.csv", index=False
)

print("✅ Saved strategy_input_data.csv with 4 required columns.")

In [ ]:
import pandas as pd
import numpy as np

# === Step 1: Load input data (train sample only) ===
# This file must include: 'target', 'predicted_PD', 'B_Ave', 'S_Ave'
data1 = pd.read_csv("C:/Users/ganes/Desktop/AML/strategy_input_data-Train.csv")

# === Step 2: Define evaluation function ===
def evaluate_strategy(data1, target_col, pd_col, balance_col, spend_col, threshold):
    """
    Returns default rate and expected annual revenue for a given threshold.
    """
    accepted = data1[data1[pd_col] < threshold]
    
    if accepted.empty:
        return 0.0, 0.0

    default_rate = accepted[target_col].mean()

    # Revenue only from non-defaulters
    non_defaults = accepted[accepted[target_col] == 0]
    monthly_revenue = non_defaults[balance_col] * 0.02 + non_defaults[spend_col] * 0.001
    annual_revenue = monthly_revenue.sum() * 12

    return round(default_rate, 4), round(annual_revenue, 2)

# === Step 3: Loop through thresholds and collect results ===
thresholds = np.arange(0.1, 1.01, 0.1)
results = []

for t in thresholds:
    d_rate, revenue = evaluate_strategy(
        data1=data1,
        target_col="target",
        pd_col="predicted_PD",
        balance_col="B_Ave",
        spend_col="S_Ave",
        threshold=t
    )
    results.append({
        "Threshold": round(t, 2),
        "# Total": data1[data1["predicted_PD"] < t].shape[0],
        "Default Rate": d_rate,
        "Expected Revenue": revenue
    })

# === Step 4: Convert to DataFrame and filter valid strategies ===
results_df = pd.DataFrame(results)
valid = results_df[results_df["Default Rate"] <= 0.10]

# Pick strategies
conservative = valid.iloc[0] if not valid.empty else None
aggressive = valid.iloc[-1] if not valid.empty else None

# === Step 5: Save results ===
results_df.to_csv("C:/Users/ganes/Desktop/AML/strategy_threshold_results-Train.csv", index=False)

# === Step 6: Print summary ===
print("\n Strategy Threshold Evaluation")
print(results_df)

print("\n✅ Conservative Strategy (First with default ≤ 10%):")
print(conservative)

print("\n✅ Aggressive Strategy (Last with default ≤ 10%):")
print(aggressive)

In [ ]:
import pandas as pd
import numpy as np
import joblib

# === Step 1: Load your merged data ===
df = pd.read_csv("C:/Users/ganes/Desktop/AML/final_data.csv")
df["S_2"] = pd.to_datetime(df["S_2"])

# === Step 2: Filter for last 6 months (Nov 2017 to Apr 2018) ===
last6 = df[(df["S_2"] >= "2017-10-01") & (df["S_2"] <= "2018-04-01")]

# === Step 3: Calculate S_Ave and B_Ave ===
avg_df = last6.groupby("customer_ID")[["S_25", "B_1"]].mean().reset_index()
avg_df.columns = ["customer_ID", "S_Ave", "B_Ave"]

# === Step 4: Load your Step6 data to get target ===
step6 = pd.read_csv("C:/Users/ganes/Desktop/AML/test1_set.csv")
targets = step6[["customer_ID", "target"]]

# === Step 5: Load selected features and model ===
selected_features = pd.read_csv("C:/Users/ganes/Desktop/AML/train_selected_features.csv", nrows=1).columns.tolist()
xgb_model = joblib.load("C:/Users/ganes/Desktop/AML/final_model.xgb")

X = step6[selected_features]

customer_ids = step6["customer_ID"]
y = step6["target"]
preds = xgb_model.predict_proba(X)[:, 1]

# === Step 6: Create predictions DataFrame ===
pred_df = pd.DataFrame({
    "customer_ID": customer_ids,
    "predicted_PD": preds
})

# === Step 7: Merge predictions + target + average spend/balance ===
final = pred_df.merge(targets, on="customer_ID").merge(avg_df, on="customer_ID")

# === Step 8: Save the clean strategy input file ===
final[["target", "predicted_PD", "B_Ave", "S_Ave"]].to_csv(
    "C:/Users/ganes/Desktop/AML/strategy_input_data-Test1.csv", index=False
)

print("✅ Saved strategy_input_data.csv with 4 required columns.")

In [ ]:
import pandas as pd
import numpy as np

# === Step 1: Load input data (train sample only) ===
# This file must include: 'target', 'predicted_PD', 'B_Ave', 'S_Ave'
data1 = pd.read_csv("C:/Users/ganes/Desktop/AML/strategy_input_data-Test1.csv")

# === Step 2: Define evaluation function ===
def evaluate_strategy(data1, target_col, pd_col, balance_col, spend_col, threshold):
    """
    Returns default rate and expected annual revenue for a given threshold.
    """
    accepted = data1[data1[pd_col] < threshold]
    
    if accepted.empty:
        return 0.0, 0.0

    default_rate = accepted[target_col].mean()

    # Revenue only from non-defaulters
    non_defaults = accepted[accepted[target_col] == 0]
    monthly_revenue = non_defaults[balance_col] * 0.02 + non_defaults[spend_col] * 0.001
    annual_revenue = monthly_revenue.sum() * 12

    return round(default_rate, 4), round(annual_revenue, 2)

# === Step 3: Loop through thresholds and collect results ===
thresholds = np.arange(0.1, 1.01, 0.1)
results = []

for t in thresholds:
    d_rate, revenue = evaluate_strategy(
        data1=data1,
        target_col="target",
        pd_col="predicted_PD",
        balance_col="B_Ave",
        spend_col="S_Ave",
        threshold=t
    )
    results.append({
        "Threshold": round(t, 2),
        "# Total": data1[data1["predicted_PD"] < t].shape[0],
        "Default Rate": d_rate,
        "Expected Revenue": revenue
    })

# === Step 4: Convert to DataFrame and filter valid strategies ===
results_df = pd.DataFrame(results)
valid = results_df[results_df["Default Rate"] <= 0.10]

# Pick strategies
conservative = valid.iloc[0] if not valid.empty else None
aggressive = valid.iloc[-1] if not valid.empty else None

# === Step 5: Save results ===
results_df.to_csv("C:/Users/ganes/Desktop/AML/strategy_threshold_results-Test1.csv", index=False)

# === Step 6: Print summary ===
print("\n Strategy Threshold Evaluation")
print(results_df)

print("\n✅ Conservative Strategy (First with default ≤ 10%):")
print(conservative)

print("\n✅ Aggressive Strategy (Last with default ≤ 10%):")
print(aggressive)

In [ ]:
import pandas as pd
import numpy as np
import joblib

# === Step 1: Load your merged data ===
df = pd.read_csv("C:/Users/ganes/Desktop/AML/final_data.csv")
df["S_2"] = pd.to_datetime(df["S_2"])

# === Step 2: Filter for last 6 months (Nov 2017 to Apr 2018) ===
last6 = df[(df["S_2"] >= "2017-10-01") & (df["S_2"] <= "2018-04-01")]

# === Step 3: Calculate S_Ave and B_Ave ===
avg_df = last6.groupby("customer_ID")[["S_25", "B_1"]].mean().reset_index()
avg_df.columns = ["customer_ID", "S_Ave", "B_Ave"]

# === Step 4: Load your Step6 data to get target ===
step6 = pd.read_csv("C:/Users/ganes/Desktop/AML/test2_set.csv")
targets = step6[["customer_ID", "target"]]

# === Step 5: Load selected features and model ===
selected_features = pd.read_csv("C:/Users/ganes/Desktop/AML/train_selected_features.csv", nrows=1).columns.tolist()
xgb_model = joblib.load("C:/Users/ganes/Desktop/AML/final_model.xgb")

X = step6[selected_features]

customer_ids = step6["customer_ID"]
y = step6["target"]
preds = xgb_model.predict_proba(X)[:, 1]

# === Step 6: Create predictions DataFrame ===
pred_df = pd.DataFrame({
    "customer_ID": customer_ids,
    "predicted_PD": preds
})

# === Step 7: Merge predictions + target + average spend/balance ===
final = pred_df.merge(targets, on="customer_ID").merge(avg_df, on="customer_ID")

# === Step 8: Save the clean strategy input file ===
final[["target", "predicted_PD", "B_Ave", "S_Ave"]].to_csv(
    "C:/Users/ganes/Desktop/AML/strategy_input_data-Test2.csv", index=False
)

print("✅ Saved strategy_input_data.csv with 4 required columns.")

In [ ]:
# === Step 1: Load input data (train sample only) ===
# This file must include: 'target', 'predicted_PD', 'B_Ave', 'S_Ave'
import pandas as pd
import numpy as np
data2 = pd.read_csv("C:/Users/ganes/Desktop/AML/strategy_input_data-Test2.csv")

# === Step 2: Define evaluation function ===
def evaluate_strategy(data2, target_col, pd_col, balance_col, spend_col, threshold):
    """
    Returns default rate and expected annual revenue for a given threshold.
    """
    accepted = data2[data2[pd_col] < threshold]
    
    if accepted.empty:
        return 0.0, 0.0

    default_rate = accepted[target_col].mean()

    # Revenue only from non-defaulters
    non_defaults = accepted[accepted[target_col] == 0]
    monthly_revenue = non_defaults[balance_col] * 0.02 + non_defaults[spend_col] * 0.001
    annual_revenue = monthly_revenue.sum() * 12

    return round(default_rate, 4), round(annual_revenue, 2)

# === Step 3: Loop through thresholds and collect results ===
thresholds = np.arange(0.1, 1.01, 0.1)
results = []

for t in thresholds:
    d_rate, revenue = evaluate_strategy(
        data2=data2,
        target_col="target",
        pd_col="predicted_PD",
        balance_col="B_Ave",
        spend_col="S_Ave",
        threshold=t
    )
    results.append({
        "Threshold": round(t, 2),
        "# Total": data2[data2["predicted_PD"] < t].shape[0],
        "Default Rate": d_rate,
        "Expected Revenue": revenue
    })

# === Step 4: Convert to DataFrame and filter valid strategies ===
results_df = pd.DataFrame(results)
valid = results_df[results_df["Default Rate"] <= 0.10]

# Pick strategies
conservative = valid.iloc[0] if not valid.empty else None
aggressive = valid.iloc[-1] if not valid.empty else None

# === Step 5: Save results ===
results_df.to_csv("C:/Users/ganes/Desktop/AML/strategy_threshold_results-Test2.csv", index=False)

# === Step 6: Print summary ===
print("\n Strategy Threshold Evaluation")
print(results_df)

print("\n✅ Conservative Strategy (First with default ≤ 10%):")
print(conservative)

print("\n✅ Aggressive Strategy (Last with default ≤ 10%):")
print(aggressive)

In [ ]:
import pandas as pd

# Load your results file
file_path = "C:/Users/ganes/Desktop/AML/nn_grid_results_with_avg_std.csv"
df = pd.read_csv(file_path)

# Sort by average AUC descending, then by std deviation ascending
best_model = df.sort_values(by=["AUC_Avg", "AUC_Std"], ascending=[False, True]).iloc[0]

# Display the best model
print("✅ Best Neural Network Model:")
print(best_model)

# Optional: Save the best model details to a separate CSV
best_model.to_frame().T.to_csv("C:/Users/ganes/Desktop/AML/best_nn_model.csv", index=False)


In [ ]:
import pandas as pd
import numpy as np
import joblib

# === Step 1: Load merged data ===
df = pd.read_csv("C:/Users/ganes/Desktop/AML/final_data.csv")
df["S_2"] = pd.to_datetime(df["S_2"])

# === Step 2: Filter for last 6 months (Nov 2017 to Apr 2018) ===
last6 = df[(df["S_2"] >= "2017-10-01") & (df["S_2"] <= "2018-04-01")]

# === Step 3: Calculate S_Ave and B_Ave ===
avg_df = last6.groupby("customer_ID")[["S_25", "B_1"]].mean().reset_index()
avg_df.columns = ["customer_ID", "S_Ave", "B_Ave"]

# === Step 4: Load Train set and Target ===
train = pd.read_csv("C:/Users/ganes/Desktop/AML/train_set.csv")
targets = train[["customer_ID", "target"]]

# === Step 5: Load selected features and model ===
selected_features = pd.read_csv("C:/Users/ganes/Desktop/AML/train_selected_features.csv", nrows=1).columns.tolist()
xgb_model = joblib.load("C:/Users/ganes/Desktop/AML/final_model.xgb")

X = train[selected_features]
customer_ids = train["customer_ID"]
y = train["target"]
preds = xgb_model.predict_proba(X)[:, 1]

# === Step 6: Create predictions DataFrame ===
pred_df = pd.DataFrame({
    "customer_ID": customer_ids,
    "predicted_PD": preds
})

# === Step 7: Merge predictions + target + average spend/balance ===
final = pred_df.merge(targets, on="customer_ID").merge(avg_df, on="customer_ID")

# === Step 8: Save the clean strategy input file for train ===
final[["target", "predicted_PD", "B_Ave", "S_Ave"]].to_csv(
    "C:/Users/ganes/Desktop/AML/strategy_input_data-Train-loss.csv", index=False
)

print("✅ strategy_input_data-Train.csv created with required 4 columns.")

# === Step 9: Evaluation with Stop Loss ===
def evaluate_strategy_with_stop_loss(data, target_col, pd_col, balance_col, spend_col, threshold, stop_loss_pct=0.10):
    accepted = data[data[pd_col] < threshold]

    if accepted.empty:
        return 0.0, 0.0, 0.0, 0.0, False

    defaulters = accepted[accepted[target_col] == 1]
    non_defaulters = accepted[accepted[target_col] == 0]

    monthly_revenue = non_defaulters[balance_col] * 0.02 + non_defaulters[spend_col] * 0.001
    total_revenue = monthly_revenue.sum() * 12

    total_loss = defaulters[balance_col].sum()
    default_rate = accepted[target_col].mean()
    profit = total_revenue - total_loss

    total_balance = accepted[balance_col].sum()
    is_valid = (total_loss / total_balance) <= stop_loss_pct if total_balance > 0 else False

    return round(default_rate, 4), round(total_revenue, 2), round(total_loss, 2), round(profit, 2), is_valid

# === Step 10: Run strategy evaluations for multiple thresholds ===
data_train = final[["target", "predicted_PD", "B_Ave", "S_Ave"]]
thresholds = np.arange(0.1, 1.01, 0.1)
results = []

for t in thresholds:
    d_rate, revenue, loss, profit, valid = evaluate_strategy_with_stop_loss(
        data=data_train,
        target_col="target",
        pd_col="predicted_PD",
        balance_col="B_Ave",
        spend_col="S_Ave",
        threshold=t,
        stop_loss_pct=0.10
    )
    results.append({
        "Threshold": round(t, 2),
        "# Total Accepted": data_train[data_train["predicted_PD"] < t].shape[0],
        "Default Rate": d_rate,
        "Revenue": revenue,
        "Loss": loss,
        "Profit": profit,
        "Valid (Stop Loss ≤ 10%)": valid
    })

# === Step 11: Save and print ===
strategy_df = pd.DataFrame(results)
strategy_df.to_csv("C:/Users/ganes/Desktop/AML/strategy_threshold_results-Train-Loss.csv", index=False)

print("\n Strategy Threshold Evaluation for Train")
print(strategy_df)

valid_strategies = strategy_df[strategy_df["Default Rate"] <= 0.10]
conservative = valid_strategies.iloc[0] if not valid_strategies.empty else None
aggressive = valid_strategies.iloc[-1] if not valid_strategies.empty else None

print("\n✅ Conservative Strategy:")
print(conservative)
print("\n✅ Aggressive Strategy:")
print(aggressive)


In [ ]:
import pandas as pd
import numpy as np
import joblib

# === Step 1: Load merged data ===
df = pd.read_csv("C:/Users/ganes/Desktop/AML/final_data.csv")
df["S_2"] = pd.to_datetime(df["S_2"])

# === Step 2: Filter for last 6 months (Nov 2017 to Apr 2018) ===
last6 = df[(df["S_2"] >= "2017-10-01") & (df["S_2"] <= "2018-04-01")]

# === Step 3: Calculate S_Ave and B_Ave ===
avg_df = last6.groupby("customer_ID")[["S_25", "B_1"]].mean().reset_index()
avg_df.columns = ["customer_ID", "S_Ave", "B_Ave"]

# === Step 4: Load Train set and Target ===
train = pd.read_csv("C:/Users/ganes/Desktop/AML/test1_set.csv")
targets = train[["customer_ID", "target"]]

# === Step 5: Load selected features and model ===
selected_features = pd.read_csv("C:/Users/ganes/Desktop/AML/train_selected_features.csv", nrows=1).columns.tolist()
xgb_model = joblib.load("C:/Users/ganes/Desktop/AML/final_model.xgb")

X = train[selected_features]
customer_ids = train["customer_ID"]
y = train["target"]
preds = xgb_model.predict_proba(X)[:, 1]

# === Step 6: Create predictions DataFrame ===
pred_df = pd.DataFrame({
    "customer_ID": customer_ids,
    "predicted_PD": preds
})

# === Step 7: Merge predictions + target + average spend/balance ===
final = pred_df.merge(targets, on="customer_ID").merge(avg_df, on="customer_ID")

# === Step 8: Save the clean strategy input file for train ===
final[["target", "predicted_PD", "B_Ave", "S_Ave"]].to_csv(
    "C:/Users/ganes/Desktop/AML/strategy_input_data-Test1-loss.csv", index=False
)

print("✅ strategy_input_data-Train.csv created with required 4 columns.")

# === Step 9: Evaluation with Stop Loss ===
def evaluate_strategy_with_stop_loss(data, target_col, pd_col, balance_col, spend_col, threshold, stop_loss_pct=0.10):
    accepted = data[data[pd_col] < threshold]

    if accepted.empty:
        return 0.0, 0.0, 0.0, 0.0, False

    defaulters = accepted[accepted[target_col] == 1]
    non_defaulters = accepted[accepted[target_col] == 0]

    monthly_revenue = non_defaulters[balance_col] * 0.02 + non_defaulters[spend_col] * 0.001
    total_revenue = monthly_revenue.sum() * 12

    total_loss = defaulters[balance_col].sum()
    default_rate = accepted[target_col].mean()
    profit = total_revenue - total_loss

    total_balance = accepted[balance_col].sum()
    is_valid = (total_loss / total_balance) <= stop_loss_pct if total_balance > 0 else False

    return round(default_rate, 4), round(total_revenue, 2), round(total_loss, 2), round(profit, 2), is_valid

# === Step 10: Run strategy evaluations for multiple thresholds ===
data_train = final[["target", "predicted_PD", "B_Ave", "S_Ave"]]
thresholds = np.arange(0.1, 1.01, 0.1)
results = []

for t in thresholds:
    d_rate, revenue, loss, profit, valid = evaluate_strategy_with_stop_loss(
        data=data_train,
        target_col="target",
        pd_col="predicted_PD",
        balance_col="B_Ave",
        spend_col="S_Ave",
        threshold=t,
        stop_loss_pct=0.10
    )
    results.append({
        "Threshold": round(t, 2),
        "# Total Accepted": data_train[data_train["predicted_PD"] < t].shape[0],
        "Default Rate": d_rate,
        "Revenue": revenue,
        "Loss": loss,
        "Profit": profit,
        "Valid (Stop Loss ≤ 10%)": valid
    })

# === Step 11: Save and print ===
strategy_df = pd.DataFrame(results)
strategy_df.to_csv("C:/Users/ganes/Desktop/AML/strategy_threshold_results-Test1-Loss.csv", index=False)

print("\n Strategy Threshold Evaluation for Test1_set")
print(strategy_df)

valid_strategies = strategy_df[strategy_df["Default Rate"] <= 0.10]
conservative = valid_strategies.iloc[0] if not valid_strategies.empty else None
aggressive = valid_strategies.iloc[-1] if not valid_strategies.empty else None

print("\n✅ Conservative Strategy:")
print(conservative)
print("\n✅ Aggressive Strategy:")
print(aggressive)

In [ ]:
import pandas as pd
import numpy as np
import joblib

# === Step 1: Load merged data ===
df = pd.read_csv("C:/Users/ganes/Desktop/AML/final_data.csv")
df["S_2"] = pd.to_datetime(df["S_2"])

# === Step 2: Filter for last 6 months (Nov 2017 to Apr 2018) ===
last6 = df[(df["S_2"] >= "2017-10-01") & (df["S_2"] <= "2018-04-01")]

# === Step 3: Calculate S_Ave and B_Ave ===
avg_df = last6.groupby("customer_ID")[["S_25", "B_1"]].mean().reset_index()
avg_df.columns = ["customer_ID", "S_Ave", "B_Ave"]

# === Step 4: Load Train set and Target ===
train = pd.read_csv("C:/Users/ganes/Desktop/AML/test2_set.csv")
targets = train[["customer_ID", "target"]]

# === Step 5: Load selected features and model ===
selected_features = pd.read_csv("C:/Users/ganes/Desktop/AML/train_selected_features.csv", nrows=1).columns.tolist()
xgb_model = joblib.load("C:/Users/ganes/Desktop/AML/final_model.xgb")

X = train[selected_features]
customer_ids = train["customer_ID"]
y = train["target"]
preds = xgb_model.predict_proba(X)[:, 1]

# === Step 6: Create predictions DataFrame ===
pred_df = pd.DataFrame({
    "customer_ID": customer_ids,
    "predicted_PD": preds
})

# === Step 7: Merge predictions + target + average spend/balance ===
final = pred_df.merge(targets, on="customer_ID").merge(avg_df, on="customer_ID")

# === Step 8: Save the clean strategy input file for train ===
final[["target", "predicted_PD", "B_Ave", "S_Ave"]].to_csv(
    "C:/Users/ganes/Desktop/AML/strategy_input_data-Test2-loss.csv", index=False
)

print("✅ strategy_input_data-Train.csv created with required 4 columns.")

# === Step 9: Evaluation with Stop Loss ===
def evaluate_strategy_with_stop_loss(data, target_col, pd_col, balance_col, spend_col, threshold, stop_loss_pct=0.10):
    accepted = data[data[pd_col] < threshold]

    if accepted.empty:
        return 0.0, 0.0, 0.0, 0.0, False

    defaulters = accepted[accepted[target_col] == 1]
    non_defaulters = accepted[accepted[target_col] == 0]

    monthly_revenue = non_defaulters[balance_col] * 0.02 + non_defaulters[spend_col] * 0.001
    total_revenue = monthly_revenue.sum() * 12

    total_loss = defaulters[balance_col].sum()
    default_rate = accepted[target_col].mean()
    profit = total_revenue - total_loss

    total_balance = accepted[balance_col].sum()
    is_valid = (total_loss / total_balance) <= stop_loss_pct if total_balance > 0 else False

    return round(default_rate, 4), round(total_revenue, 2), round(total_loss, 2), round(profit, 2), is_valid

# === Step 10: Run strategy evaluations for multiple thresholds ===
data_train = final[["target", "predicted_PD", "B_Ave", "S_Ave"]]
thresholds = np.arange(0.1, 1.01, 0.1)
results = []

for t in thresholds:
    d_rate, revenue, loss, profit, valid = evaluate_strategy_with_stop_loss(
        data=data_train,
        target_col="target",
        pd_col="predicted_PD",
        balance_col="B_Ave",
        spend_col="S_Ave",
        threshold=t,
        stop_loss_pct=0.10
    )
    results.append({
        "Threshold": round(t, 2),
        "# Total Accepted": data_train[data_train["predicted_PD"] < t].shape[0],
        "Default Rate": d_rate,
        "Revenue": revenue,
        "Loss": loss,
        "Profit": profit,
        "Valid (Stop Loss ≤ 10%)": valid
    })

# === Step 11: Save and print ===
strategy_df = pd.DataFrame(results)
strategy_df.to_csv("C:/Users/ganes/Desktop/AML/strategy_threshold_results-Test2-Loss.csv", index=False)

print("\n Strategy Threshold Evaluation for Test2_Set")
print(strategy_df)

valid_strategies = strategy_df[strategy_df["Default Rate"] <= 0.10]
conservative = valid_strategies.iloc[0] if not valid_strategies.empty else None
aggressive = valid_strategies.iloc[-1] if not valid_strategies.empty else None

print("\n✅ Conservative Strategy:")
print(conservative)
print("\n✅ Aggressive Strategy:")
print(aggressive)